# 03 · Indicadores clave y hallazgos (SQL sobre el modelo estrella)
**Objetivo:** calcular el OTD por zona, transportista y tipo de servicio, el retraso promedio, la tasa de devolución y el costo por número de intentos. Identificar la peor zona, el peor transportista, el día con mayor retraso y el sobrecosto.

**Entrada:** `bd/courier_dw.sqlite` (generada por `02_etl.ipynb`)

**Definición:** OTD = envíos entregados en o antes de la fecha promesa ÷ envíos entregados.

In [1]:
import sqlite3
import pandas as pd

con = sqlite3.connect("bd/courier_dw.sqlite")

def sql(consulta):
    return pd.read_sql(consulta, con)

sql("SELECT name FROM sqlite_master WHERE type = 'table'")

,name
0,dim_tiempo
1,dim_zona
2,dim_transportista
3,dim_servicio
4,fact_envios


# Parte A · Indicadores

## A1. Indicadores generales

In [2]:
sql("""
SELECT COUNT(*)                                                AS envios,
       ROUND(100.0 * SUM(flag_a_tiempo) / SUM(flag_entregado), 2) AS otd_pct,
       ROUND(AVG(dias_retraso), 2)                             AS retraso_neto_dias,
       ROUND(AVG(CASE WHEN dias_retraso > 0 THEN dias_retraso END), 2) AS retraso_tardios_dias,
       ROUND(100.0 * SUM(flag_devuelto) / COUNT(*), 2)         AS tasa_devolucion_pct,
       ROUND(AVG(costo_envio), 2)                              AS costo_promedio,
       ROUND(SUM(costo_envio), 2)                              AS costo_total
FROM fact_envios
""")

,envios,otd_pct,retraso_neto_dias,retraso_tardios_dias,tasa_devolucion_pct,costo_promedio,costo_total
0,41952,74.52,-0.09,1.56,1.7,3.88,162906.5


1 de cada 4 envíos llega tarde; cuando se retrasa, llega en promedio 1,56 días tarde.

## A2. OTD por zona

In [3]:
sql("""
SELECT z.zona, COUNT(*) AS envios,
       ROUND(100.0 * SUM(f.flag_a_tiempo) / SUM(f.flag_entregado), 1) AS otd_pct,
       ROUND(AVG(f.dias_retraso), 2)                     AS retraso_neto,
       ROUND(100.0 * SUM(f.flag_devuelto) / COUNT(*), 2) AS devolucion_pct
FROM fact_envios f JOIN dim_zona z ON z.id_zona = f.id_zona
GROUP BY z.zona ORDER BY otd_pct
""")

,zona,envios,otd_pct,retraso_neto,devolucion_pct
0,Rural,4145,19.4,1.61,2.03
1,Valles,5953,58.3,0.40,1.58
2,Sur,10084,79.9,-0.22,1.39
3,Centro,9912,86.2,-0.44,1.77
4,Norte,11858,87.6,-0.52,1.86


## A3. OTD por transportista

In [4]:
sql("""
SELECT t.codigo_transportista AS transportista, COUNT(*) AS envios,
       ROUND(100.0 * SUM(f.flag_a_tiempo) / SUM(f.flag_entregado), 1) AS otd_pct,
       ROUND(AVG(f.dias_retraso), 2)                     AS retraso_neto,
       ROUND(100.0 * SUM(f.flag_devuelto) / COUNT(*), 2) AS devolucion_pct
FROM fact_envios f JOIN dim_transportista t ON t.id_transportista = f.id_transportista
GROUP BY t.codigo_transportista ORDER BY otd_pct
""")

,transportista,envios,otd_pct,retraso_neto,devolucion_pct
0,TR07,5252,45.2,0.80,1.69
1,TR04,5281,73.1,-0.04,1.69
2,TR01,5191,77.6,-0.17,1.96
3,TR03,5283,78.5,-0.20,1.86
4,TR02,5231,79.8,-0.26,1.74
5,TR08,5221,79.8,-0.24,1.49
6,TR06,5249,80.3,-0.26,1.47
7,TR05,5244,81.9,-0.33,1.70


## A4. OTD por tipo de servicio

In [5]:
sql("""
SELECT s.tipo_servicio, COUNT(*) AS envios,
       ROUND(100.0 * SUM(f.flag_a_tiempo) / SUM(f.flag_entregado), 1) AS otd_pct,
       ROUND(AVG(f.dias_retraso), 2)                     AS retraso_neto,
       ROUND(100.0 * SUM(f.flag_devuelto) / COUNT(*), 2) AS devolucion_pct
FROM fact_envios f JOIN dim_servicio s ON s.id_servicio = f.id_servicio
GROUP BY s.tipo_servicio
""")

,tipo_servicio,envios,otd_pct,retraso_neto,devolucion_pct
0,Estándar,29369,74.4,-0.29,1.69
1,Express,12583,74.7,0.39,1.72


El OTD es casi igual en ambos servicios: el tipo de servicio no explica los retrasos.

## A5. Costo promedio según número de intentos

In [6]:
sql("""
SELECT intentos_entrega, COUNT(*) AS envios,
       ROUND(AVG(costo_envio), 2) AS costo_promedio,
       ROUND(SUM(costo_envio), 2) AS costo_total
FROM fact_envios GROUP BY intentos_entrega ORDER BY intentos_entrega
""")

,intentos_entrega,envios,costo_promedio,costo_total
0,1,34882,3.55,123924.47
1,2,5018,5.06,25400.58
2,3,2052,6.62,13581.45


Cada intento adicional sube el costo en unos USD 1,50.

# Parte B · Hallazgos

## B1. Peor zona y peor transportista

In [7]:
peor_zona = sql("""
SELECT z.zona, ROUND(100.0 * SUM(f.flag_a_tiempo) / SUM(f.flag_entregado), 1) AS otd_pct
FROM fact_envios f JOIN dim_zona z ON z.id_zona = f.id_zona
GROUP BY z.zona ORDER BY otd_pct LIMIT 1""")

peor_tr = sql("""
SELECT t.codigo_transportista AS transportista, ROUND(100.0 * SUM(f.flag_a_tiempo) / SUM(f.flag_entregado), 1) AS otd_pct
FROM fact_envios f JOIN dim_transportista t ON t.id_transportista = f.id_transportista
GROUP BY t.codigo_transportista ORDER BY otd_pct LIMIT 1""")

print(f"Peor zona:          {peor_zona.zona[0]}  (OTD {peor_zona.otd_pct[0]} %)")
print(f"Peor transportista: {peor_tr.transportista[0]}   (OTD {peor_tr.otd_pct[0]} %)")

Peor zona:          Rural  (OTD 19.4 %)
Peor transportista: TR07   (OTD 45.2 %)


## B2. ¿Por qué Rural? Tránsito frente a la promesa

In [8]:
sql("""
SELECT z.zona,
       ROUND(AVG(f.dias_transito), 2)    AS transito_promedio_dias,
       ROUND(AVG(f.intentos_entrega), 2) AS intentos_promedio
FROM fact_envios f JOIN dim_zona z ON z.id_zona = f.id_zona
GROUP BY z.zona ORDER BY transito_promedio_dias DESC
""")

,zona,transito_promedio_dias,intentos_promedio
0,Rural,4.01,1.47
1,Valles,2.79,1.18
2,Sur,2.17,1.19
3,Centro,1.97,1.19
4,Norte,1.89,1.19


Rural tarda unos 4 días frente a una promesa de 3: el incumplimiento es estructural.

## B3. ¿TR07 es malo en todas las zonas?

In [9]:
sql("""
SELECT z.zona,
       ROUND(100.0 * SUM(CASE WHEN t.codigo_transportista = 'TR07' THEN f.flag_a_tiempo END)
             / SUM(CASE WHEN t.codigo_transportista = 'TR07' THEN f.flag_entregado END), 1) AS otd_TR07,
       ROUND(100.0 * SUM(CASE WHEN t.codigo_transportista <> 'TR07' THEN f.flag_a_tiempo END)
             / SUM(CASE WHEN t.codigo_transportista <> 'TR07' THEN f.flag_entregado END), 1) AS otd_resto
FROM fact_envios f
JOIN dim_zona z ON z.id_zona = f.id_zona
JOIN dim_transportista t ON t.id_transportista = f.id_transportista
GROUP BY z.zona ORDER BY otd_resto
""")

,zona,otd_TR07,otd_resto
0,Rural,2.8,21.8
1,Valles,21.4,63.7
2,Sur,45.8,84.8
3,Centro,59.1,89.9
4,Norte,60.3,91.5


TR07 queda unos 30 puntos por debajo en todas las zonas: el problema es del transportista.

## B4. Día de la semana con mayor retraso (por fecha de recepción)

In [10]:
sql("""
SELECT d.num_dia_semana, d.dia_semana, COUNT(*) AS envios,
       ROUND(100.0 * SUM(f.flag_a_tiempo) / SUM(f.flag_entregado), 1) AS otd_pct,
       ROUND(AVG(f.dias_retraso), 2) AS retraso_neto,
       ROUND(AVG(f.dias_atraso), 2)  AS atraso_por_envio
FROM fact_envios f JOIN dim_tiempo d ON d.id_tiempo = f.id_tiempo_recepcion
GROUP BY d.num_dia_semana, d.dia_semana ORDER BY d.num_dia_semana
""")

,num_dia_semana,dia_semana,envios,otd_pct,retraso_neto,atraso_por_envio
0,1,Lunes,7358,62.8,0.31,0.61
1,2,Martes,7294,77.3,-0.18,0.35
2,3,Miércoles,6865,76.8,-0.17,0.35
3,4,Jueves,7033,77.8,-0.19,0.34
4,5,Viernes,7339,77.2,-0.18,0.34
5,6,Sábado,3080,74.9,-0.11,0.38
6,7,Domingo,2983,76.9,-0.15,0.36


## B5. ¿El efecto lunes ocurre en todas las zonas?

In [11]:
lunes = sql("""
SELECT z.zona, d.num_dia_semana,
       ROUND(100.0 * SUM(f.flag_a_tiempo) / SUM(f.flag_entregado), 0) AS otd_pct
FROM fact_envios f
JOIN dim_tiempo d ON d.id_tiempo = f.id_tiempo_recepcion
JOIN dim_zona z  ON z.id_zona  = f.id_zona
GROUP BY z.zona, d.num_dia_semana""")
tabla = lunes.pivot(index="zona", columns="num_dia_semana", values="otd_pct")
tabla.columns = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
tabla

,Lun,Mar,Mié,Jue,Vie,Sáb,Dom
zona,,,,,,,
Centro,76.0,88.0,89.0,90.0,87.0,87.0,88.0
Norte,78.0,90.0,89.0,90.0,89.0,88.0,89.0
Rural,10.0,23.0,22.0,19.0,23.0,22.0,18.0
Sur,66.0,83.0,81.0,84.0,83.0,80.0,85.0
Valles,39.0,63.0,62.0,63.0,63.0,59.0,62.0


El lunes es el peor día en todas las zonas: es un problema de capacidad operativa del lunes.

## B6. Sobrecosto por intentos adicionales y devoluciones

In [12]:
sql("""
SELECT SUM(intentos_adicionales)                                 AS intentos_adicionales,
       ROUND(SUM(sobrecosto_intentos), 2)                        AS sobrecosto_intentos,
       SUM(flag_devuelto)                                        AS devoluciones,
       ROUND(SUM(sobrecosto_devolucion), 2)                      AS sobrecosto_devoluciones,
       ROUND(SUM(sobrecosto_intentos) + SUM(sobrecosto_devolucion), 2) AS sobrecosto_total,
       ROUND(100.0 * (SUM(sobrecosto_intentos) + SUM(sobrecosto_devolucion)) / SUM(costo_envio), 1) AS pct_del_costo_total,
       ROUND(SUM(CASE WHEN flag_devuelto = 1 THEN costo_envio END), 2) AS costo_total_devueltos
FROM fact_envios
""")

,intentos_adicionales,sobrecosto_intentos,devoluciones,sobrecosto_devoluciones,sobrecosto_total,pct_del_costo_total,costo_total_devueltos
0,9122,13665.54,713,1423.23,15088.77,9.3,4430.89


## B7. ¿Dónde se concentra el sobrecosto?

In [13]:
sql("""
SELECT z.zona, COUNT(*) AS envios,
       ROUND(SUM(f.sobrecosto_intentos + f.sobrecosto_devolucion), 0) AS sobrecosto_usd,
       ROUND(SUM(f.sobrecosto_intentos + f.sobrecosto_devolucion) / COUNT(*), 2) AS sobrecosto_por_envio
FROM fact_envios f JOIN dim_zona z ON z.id_zona = f.id_zona
GROUP BY z.zona ORDER BY sobrecosto_por_envio DESC
""")

,zona,envios,sobrecosto_usd,sobrecosto_por_envio
0,Rural,4145,3108.0,0.75
1,Centro,9912,3238.0,0.33
2,Norte,11858,3775.0,0.32
3,Valles,5953,1817.0,0.31
4,Sur,10084,3150.0,0.31


## Resumen de hallazgos
| Pregunta | Respuesta |
|---|---|
| Zona con peor puntualidad | **Rural**, 19,4 % de OTD (tránsito de 4 días frente a una promesa de 3) |
| Transportista con peor puntualidad | **TR07**, 45,2 % (el peor en todas las zonas) |
| Día con mayor retraso | **Lunes**, 62,8 % de OTD y 0,61 días de atraso por envío |
| Sobrecosto total | **USD 15 089**: 13 666 por intentos + 1 423 por devoluciones (9,3 % del costo) |